<a href="https://colab.research.google.com/github/ks21xf/4P96_A1/blob/main/4P96_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

todo
- local best instead of global best
- convergence as early stopping condition
- penalty function for infeasible solutions
- caching / parallelization / performance estimation if needed
- Multi-phase PSO
- lots of testing
- visualizations (specified in our proposal)

In [1]:
!pip install medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.2 MB/s eta 0:00:00


In [2]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

In [3]:
seed = 10

np.random.seed(seed)
torch.manual_seed(seed)
g = torch.Generator().manual_seed(seed)

In [4]:
# Load dataset
from medmnist import ChestMNIST
train_dataset = ChestMNIST(split='train', transform=transforms.ToTensor(), download=True)
val_dataset = ChestMNIST(split='val', transform=transforms.ToTensor(), download=True)
test_dataset = ChestMNIST(split='test', transform=transforms.ToTensor(), download=True)

100%|██████████| 82.8M/82.8M [00:04<00:00, 19.5MB/s]


In [5]:
# Dataset info
train_dataset

Dataset ChestMNIST of size 28 (chestmnist)
    Number of datapoints: 78468
    Root location: /root/.medmnist
    Split: train
    Task: multi-label, binary-class
    Number of channels: 1
    Meaning of labels: {'0': 'atelectasis', '1': 'cardiomegaly', '2': 'effusion', '3': 'infiltration', '4': 'mass', '5': 'nodule', '6': 'pneumonia', '7': 'pneumothorax', '8': 'consolidation', '9': 'edema', '10': 'emphysema', '11': 'fibrosis', '12': 'pleural', '13': 'hernia'}
    Number of samples: {'train': 78468, 'val': 11219, 'test': 22433}
    Description: The ChestMNIST is based on the NIH-ChestXray14 dataset, a dataset comprising 112,120 frontal-view X-Ray images of 30,805 unique patients with the text-mined 14 disease labels, which could be formulized as a multi-label binary-class classification task. We use the official data split, and resize the source images of 1×1024×1024 into 1×28×28.
    License: CC BY 4.0

In [6]:
num_classes = len(train_dataset.info['label'])
input_dim = int(np.prod(train_dataset[0][0].shape))


In [37]:
train_loader = DataLoader(train_dataset, batch_size=20)
for images, labels in train_loader:
  print(images[0].unsqueeze(1).shape)
  print(images.shape)


  print(labels[0])
  break

torch.Size([1, 1, 28, 28])
torch.Size([20, 1, 28, 28])
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


In [ ]:
search_bounds = [
    (1, 5), # number of hidden layers
    (16, 1024), # number of nodes per layer
    (0.00001, 0.1), # learning rate
    (0.0, 0.99), # momentum
    (8, 256), # batch size
    (0.01, 0.0), # weight decay
    (0.0, 0.7) # dropout rate
]

In [ ]:
class Particle:
  def __init__(self, search_bounds):
    self.search_bounds = search_bounds
    num_dims = len(search_bounds)

    self.position = np.array([np.random.uniform(low, high) for low, high in search_bounds]) # initialize position randomly within bounds
    self.velocity = np.zeros(7) # initialize velocities to zero

    self.best_position = self.position
    self.best_fitness = float('inf')

  def get_network_params(self):
    return {
        'num_hidden_layers': int(self.position[0]),
        'hidden_layer_size': int(self.position[1]),
        'learning_rate': self.position[2],
        'momentum': self.position[3],
        'batch_size': int(self.position[4]),
        'weight_decay': self.position[5],
        'dropout_rate': self.position[6]
    }


In [ ]:
class MLP(nn.Module):
  def __init__(self, input_dim, num_classes, num_layers, hidden_size, dropout_rate):
    super().__init__()
    layers = []
    next_num_features = input_dim

    for i in range(num_layers):
      layers.append(nn.Linear(next_num_features, hidden_size))
      layers.append(nn.BatchNorm1d(hidden_size))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      next_num_features = hidden_size

    layers.append(nn.Linear(hidden_size, num_classes))
    self.network = nn.Sequential(*layers)

  def forward(self, x):
    return self.network(x.view(x.size(0), -1)) # flatten and pass through network


In [ ]:
# evalutate fitness of a particle by training a neural net on the parameters
def evaluate_particle(parameters, train_dataset, val_dataset, test_dataset, input_dim, num_classes, epochs=5):
  train_loader = DataLoader(train_dataset, batch_size=parameters['batch_size'], shuffle=True, generator=g)
  val_loader = DataLoader(val_dataset, batch_size=parameters['batch_size'], generator=g)
  test_loader = DataLoader(test_dataset, batch_size=parameters['batch_size'], generator=g)

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  model = MLP(input_dim=input_dim, num_classes=num_classes, num_layers=parameters['num_hidden_layers'], hidden_size=parameters['hidden_layer_size'], dropout_rate=parameters['dropout_rate']).to(device)

  optimizer = torch.optim.SGD(model.parameters(), lr=parameters['learning_rate'])
  criterion = nn.BCEWithLogitsLoss() # BCEWithLogitsLoss is used for multi-class classification problems

  #VARS FOR MINIBATCH STORAGE
  OVERFITTING_DETECTION_PARAMETER = 10 #how far we keep a history of epochs to use to check for overfitting
  running_mean_loss = 0 #stores average loss over minibatches, so this is mean per epoch
  count = 1 #counts minibatches
  loss_history = np.zeros(OVERFITTING_DETECTION_PARAMETER)
  loss_history[loss_history == 0.0] = np.nan #do this to do mean_nan - ignores nan values, if we use zeros instead it would messup the mean

  # Training
  model.train()
  for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    for images, labels in train_loader:
      images, labels = images.to(device),  labels.float().to(device)
      optimizer.zero_grad()
      loss = criterion(model(images), labels)
      running_mean_loss = running_mean_loss + (loss.item() - running_mean_loss) / count #calculate mean of loss in the batch
      count += 1
      loss.backward()
      optimizer.step()

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
      for images, labels in val_loader:
        images, labels = images.to(device), labels.float().to(device)
        val_loss += criterion(model(images), labels).item()

    val_loss = val_loss / len(val_loader)
    loss_history[epoch%OVERFITTING_DETECTION_PARAMETER] = val_loss

    print("val",val_loss) #debug
    #check overfitting
    mean = np.nanmean(loss_history)
    std = np.nanstd(loss_history)
    if val_loss > mean + std:
        print(f"Overfitting detected at epoch {epoch}, stopping search early.")
        break

  # Testing
  model.eval()
  test_loss = 0.0
  with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.float().to(device)
        test_loss += criterion(model(images), labels).item()

  print("test",test_loss / len(test_loader)) #debug
  return test_loss / len(test_loader) # right now objective function is just the loss (for testing!)

In [ ]:
class PSO:
  def __init__(self, num_particles, search_bounds, w=0.729, c1=1.49445, c2=1.49445):
    self.particles = [Particle(search_bounds=search_bounds) for _ in range(num_particles)]
    self.global_best_position = None
    self.global_best_fitness = float('inf')
    self.w, self.c1, self.c2, = w, c1, c2

  def optimize(self, fitness_function, num_iterations):

    # Evaluate initial population
    print(f"Evaluating Initial Population")
    for i in range(len(self.particles)):
      parameters = self.particles[i].get_network_params()
      fitness = fitness_function(parameters)
      print(f"Particle {i} | Test Loss: {fitness:.4f} | Parameters: {parameters}")

      # Set Initial Personal Best
      self.particles[i].best_fitness = fitness
      self.particles[i].best_position = self.particles[i].position.copy()

      # Set Initial Global Best
      if fitness < self.global_best_fitness:
        self.global_best_position = self.particles[i].position.copy()
        self.global_best_fitness = fitness

    # Main PSO Loop
    for iteration in range(num_iterations): # TODO: add other stopping conditions
      print(f"Iteration {iteration + 1}/{num_iterations}")

      for i in range(len(self.particles)):
        # Update Velocity
        r1 = np.random.rand(len(self.particles[i].position))
        r2 = np.random.rand(len(self.particles[i].position))

        cognitive = self.c1 * r1 * (self.particles[i].best_position - self.particles[i].position)
        social = self.c2 * r2 * (self.global_best_position - self.particles[i].position)

        self.particles[i].velocity = (self.w * self.particles[i].velocity + cognitive + social)

        # Update Position
        self.particles[i].position = self.particles[i].position + self.particles[i].velocity

        # Evaluate Fitness
        parameters = self.particles[i].get_network_params()
        fitness = fitness_function(parameters)
        print(f"Particle {i+1} | Validation Loss: {fitness:.4f} | Parameters: {parameters}")

        # Update Personal Best
        if fitness < self.particles[i].best_fitness:
          self.particles[i].best_fitness = fitness
          self.particles[i].best_position = self.particles[i].position.copy()

      # Update Global Best
      # TODO: Make this local best instead!
      for particle in self.particles:
        if particle.best_fitness < self.global_best_fitness:
          self.global_best_fitness = particle.best_fitness
          self.global_best_position = particle.best_position.copy()

      best_position = self.global_best_position
      return best_position



In [ ]:
# TODO: Make this our multi-objective function instead of just
def objective_function(parameters):
  return evaluate_particle(parameters, train_dataset, val_dataset, input_dim, num_classes, epochs=5)

In [ ]:
pso = PSO(num_particles=1, search_bounds=search_bounds)
best_position = pso.optimize(objective_function, num_iterations=1)

best = Particle(search_bounds)
best.position = best_position
print("Best architecture found:", best.get_network_params())

Evaluating Initial Population
Epoch 1/5
val 0.18668449137892043
Epoch 2/5
val 0.17340315222030594
Epoch 3/5
val 0.16975394476737296
Epoch 4/5
val 0.17530909499951772
Epoch 5/5
val 0.1702138997969173
test 0.17671897302487652
Particle 0 | Validation Loss: 0.1767 | Parameters: {'num_hidden_layers': 4, 'hidden_layer_size': 901, 'learning_rate': np.float64(0.04175673929248831), 'momentum': np.float64(0.5995217887498192), 'batch_size': 135, 'weight_decay': np.float64(0.0040216335203702635), 'dropout_rate': np.float64(0.18355096279236519)}
Iteration 1/1
Epoch 1/5
val 0.18378652596757525
Epoch 2/5
val 0.17329429427073115
Epoch 3/5
val 0.1699378185328983
Epoch 4/5
val 0.17044417843932197
Epoch 5/5
val 0.1693082247816381
test 0.1757564232377949
Particle 1 | Validation Loss: 0.1758 | Parameters: {'num_hidden_layers': 4, 'hidden_layer_size': 901, 'learning_rate': np.float64(0.04175673929248831), 'momentum': np.float64(0.5995217887498192), 'batch_size': 135, 'weight_decay': np.float64(0.00402163352